# Assignment 1 — Product description evaluation

## Goal: learn LLM evauation process

Build and evaluate **short product descriptions** from structured product data: define quality **before** generating at scale, run a **baseline**, manually score a sample, try **targeted improvements**, then compare **human** ratings to an **LLM judge** with structured output.

## Business task (in my own words)

- **Input:** structured attributes per product (e.g. name, category, materials, dimensions).
- **Output:** a concise description suitable for a catalog or listing.
- **Risk:** models easily add plausible but **unsupported** details; the hardest requirement is usually **grounding** in the provided fields.

## Deliverables this notebook supports

- A clear **rubric** (Task 1) used consistently for manual eval and judge design.
- Generated descriptions and an **`assignment_01.xlsx`** export.
- A small **manual evaluation** and at least one **improvement experiment** with a short rationale.
- A **judge** with structured scores and a **comparison** to human labels.

> **Week 2 idea — rubric before generation:** If we generate before the rubric is stable, we optimize for “sounds nice” instead of measurable criteria. The rubric is the contract for what “good” means.

> **Week 1 idea — grounding:** Descriptions should only claim what the attributes support; vague or missing attributes are a signal to stay general or say “not specified,” not to invent facts.

## Setup and imports

**Environment:** Python 3.11+ with `pandas`, `openpyxl` (Excel), `openai` (API), `pydantic` (structured schemas), `tqdm` (progress). Kernel must match the env where packages are installed.

**What we import and why**

| Area | Why it matters |
|------|----------------|
| `pandas` | Load CSV, inspect rows, build the Excel artifact. |
| `openpyxl` | Write `.xlsx` for submission. |
| `openai` | Calls to the chat/completions API for generation and judging. |
| `pydantic` | Enforce **structured** judge outputs (scores + short rationale), reducing parse errors. |

**Secrets:** API keys live in environment variables (e.g. `OPENAI_API_KEY`), not in the notebook, so the notebook stays shareable and safe.

> **Week 1 — context engineering:** Later, prompts will need the **right fields** in a consistent order; setup is where we commit to reading data as the single source of truth for grounding.

> **Week 2 — evaluation-driven development:** Imports are boring on purpose: the “interesting” work is comparing runs against the **same rubric** and sample.

## Phase 0: Set up your workspace once

### Step 1: Create a project folder

Make one folder for the whole assignment, for example:

```text
module1_LLM_Evaluation_Asignment01/
```

Put inside it docs/, scripts/, data/, and models/ folders:

* the dataset CSV in data/
* your notebook in scripts/
* optional notes file in docs/

### Step 2: Create your Python environment
```bash
conda create -n assignment01 python=3.11 -y
conda activate assignment01
conda install pandas openpyxl tqdm jupyter ipykernel nbformat -y
pip install pydantic openai
```

### Step 3: Register the environment as a Jupyter kernel

This is what makes your environment show up inside the notebook UI:

```bash
python -m ipykernel install --user --name assignment01 --display-name "Python (assignment01)"
```

### Step 4: Open the folder

Then open your notebook or create a new `.ipynb`.

### Step 5: Select the kernel

In the notebook UI, choose:

```text
Python (assignment01)
```

1. **Inports**

In [1]:
import os
import re
import time
import math
import json
from typing import Literal, Optional

import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field

from openai import OpenAI

## 2.Load dataset

- upload `data/01_agents_Assignment_01_product_dataset.csv`



In [ ]:
from pathlib import Path


def _resolve_product_csv() -> Path:
    """Find the CSV whether Jupyter's cwd is the module root or scripts/."""
    name = "01_agents_Assignment_01_product_dataset.csv"
    for rel in (Path("data") / name, Path("../data") / name):
        p = rel.resolve()
        if p.is_file():
            return p
    raise FileNotFoundError(
        f"Missing {name}. Tried {Path('data') / name} and {Path('../data') / name} (cwd={Path.cwd()})"
    )


DATASET_PATH = _resolve_product_csv()
df_products = pd.read_csv(DATASET_PATH)
df_products.head()